# Minería de texto: Reconocimiento de Entidades Nombradas (NER)

## Contexto

Una empresa recibe reportes de clientes en texto libre. Cada reporte puede mencionar personas, empresas, ciudades, fechas, números de pedido e importes. El objetivo es extraer esos datos automáticamente para llenar un CRM, localizar envíos y reducir captura manual.

Este ejercicio es distinto de clasificar documentos. La clasificación responde a qué categoría pertenece un mensaje; NER responde qué información importante aparece dentro del mensaje y dónde aparece.

El dataset está incluido en este notebook, por lo que no es necesario subir archivos externos.

## Objetivos de aprendizaje

- Explicar qué es NER.
- Usar un modelo lingüístico en español.
- Extraer entidades como personas, organizaciones, lugares y fechas.
- Complementar el modelo con reglas para pedidos, teléfonos e importes.
- Convertir las entidades en una tabla estructurada.
- Interpretar resultados, errores y utilidad empresarial.

El dataset es pequeño y educativo. En producción se necesitarían datos reales, revisión humana y evaluación con ejemplos anotados.

## 1. Instalar y cargar el modelo de español

spaCy proporciona un modelo lingüístico que analiza las oraciones y reconoce entidades. La primera línea instala las bibliotecas; la segunda descarga el modelo pequeño de español en la sesión de Colab.

In [ ]:
%%capture
!pip -q install spacy pandas matplotlib seaborn
!python -m spacy download es_core_news_sm

In [ ]:
import io
import re
import pandas as pd
import spacy
import matplotlib.pyplot as plt
import seaborn as sns

nlp = spacy.load('es_core_news_sm')
pd.set_option('display.max_colwidth', 140)
print('Modelo cargado correctamente')

## 2. ¿Qué es NER?

NER significa Named Entity Recognition o Reconocimiento de Entidades Nombradas. El modelo recibe una oración y marca fragmentos que representan entidades relevantes.

Ejemplo: “María López recibió el paquete en Monterrey”. El resultado esperado sería María López como persona y Monterrey como lugar.

Una entidad tiene tres elementos: el texto detectado, su tipo y su posición dentro del mensaje. Las etiquetas pueden incluir PER para personas, ORG para organizaciones, LOC o GPE para lugares y DATE para fechas. Las etiquetas exactas dependen del modelo.

### ¿Cómo trabaja el modelo?

spaCy divide el texto en tokens, analiza la función de las palabras y utiliza patrones aprendidos de ejemplos anotados. No busca solamente palabras aisladas: también considera el contexto. Por eso el mismo término puede recibir etiquetas distintas dependiendo de la oración.

El modelo es estadístico, no infalible. Puede confundirse con nombres poco comunes, empresas nuevas, abreviaturas o frases que no aparecieron en sus datos de entrenamiento.

## 3. Dataset incluido en el notebook

La variable csv_text contiene los datos como CSV. pandas los lee desde memoria, sin depender de una ruta local. También se crea un CSV opcional para que el usuario pueda reutilizarlo.

In [ ]:
csv_text = '''id_reporte,texto
R001,María López informa que su pedido 45821 de TecnoMarket no llegó a Monterrey el 15 de agosto.
R002,José Hernández solicita una factura de $1,250.50 para Comercial del Norte.
R003,El paquete 77402 debe entregarse en Guadalajara antes del 20 de septiembre.
R004,Ana Torres reporta un cargo de 899 pesos realizado por Servicios Digitales MX.
R005,Pedro Ramírez necesita cambiar la entrega del pedido 99103 a Querétaro.
R006,Laura Gómez llamó desde el teléfono 5512345678 para consultar su factura.
R007,La empresa Innovación Logística envió el pedido 23014 a Mérida.
R008,Carlos Méndez solicita cancelar el servicio el 30 de octubre.
R009,El cliente Sofía Navarro recibió un cobro de $450.00 de Mercado Central.
R010,Roberto Díaz indica que su envío 68077 está detenido en Puebla.
R011,Grupo Alimentario del Pacífico solicita comprobante para el 5 de noviembre.
R012,El pedido 34560 de Daniela Cruz fue entregado en León.
R013,Esteban Morales reporta que la aplicación falla desde el 2 de diciembre.
R014,Mariana Silva solicita una devolución por 780 pesos a nombre de Soluciones Urbanas.
R015,El envío 81245 de Paola Reyes no aparece en el domicilio de Toluca.
R016,Raúl Castillo necesita una factura con RFC para Transportes del Bajío.
R017,El cliente Diego Vargas quiere cancelar su plan el 12 de enero.
R018,Patricia León informa que el paquete 55661 llegó incompleto a Veracruz.
R019,Alberto Ríos recibió una notificación de pago por $2,300.00.
R020,Servicios Médicos del Centro solicita localizar el pedido 77881 en Oaxaca.'''

csv_text = '\n'.join(line.replace(',', '|', 1) for line in csv_text.splitlines())
df = pd.read_csv(io.StringIO(csv_text), sep='|')
df.to_csv('reportes_clientes_ner.csv', index=False, encoding='utf-8')
print(f'Reportes: {len(df)}')
display(df.head())

### Interpretación del dataset

Un mismo reporte puede contener varias entidades. Por ejemplo, una oración puede incluir una persona, un pedido, una empresa, una ciudad y una fecha. Por eso la salida de NER puede tener varias filas por reporte.

## 4. Aplicar NER con spaCy

doc.ents contiene las entidades detectadas por el modelo. ent.text es el fragmento original, ent.label_ es su tipo y las posiciones permiten ubicarlo dentro del texto.

In [ ]:
def extraer_entidades_modelo(texto):
    doc = nlp(texto)
    return [
        {
            'entidad': ent.text,
            'tipo': ent.label_,
            'inicio': ent.start_char,
            'fin': ent.end_char,
            'origen': 'modelo_spacy'
        }
        for ent in doc.ents
    ]

df['entidades_modelo'] = df['texto'].apply(extraer_entidades_modelo)
display(df[['id_reporte', 'texto', 'entidades_modelo']].head(5))

### Cómo interpretar este resultado

Cada lista contiene las entidades reconocidas en un reporte. Una lista vacía no demuestra que no haya información útil; puede significar que el modelo no conoce el nombre, el formato o el vocabulario de la empresa.

Los modelos generales suelen reconocer personas y lugares, pero pueden no reconocer códigos internos como 45821. Por eso se agregan reglas de negocio.

### Lectura de una entidad

Si el resultado contiene entidad = Monterrey y tipo = LOC, significa que el modelo encontró exactamente ese fragmento y lo interpretó como lugar. Las posiciones inicio y fin permiten resaltarlo en una interfaz o relacionarlo con el texto sin perder el contexto.

En un sistema real conviene guardar también el nivel de confianza del modelo, cuando esté disponible, y enviar a revisión las entidades de baja confianza.

## 5. Convertir las entidades en una tabla

Una lista anidada es difícil de analizar. La siguiente celda crea una fila por entidad, conservando el reporte, el texto encontrado, el tipo y las posiciones.

In [ ]:
filas = []
for _, fila in df.iterrows():
    for entidad in fila['entidades_modelo']:
        filas.append({'id_reporte': fila['id_reporte'], **entidad})

entidades_df = pd.DataFrame(filas)
display(entidades_df.head(12))
print(f'Entidades detectadas por spaCy: {len(entidades_df)}')

In [ ]:
conteo = entidades_df['tipo'].value_counts()
display(conteo.to_frame('cantidad'))
sns.barplot(x=conteo.values, y=conteo.index)
plt.title('Entidades detectadas por tipo')
plt.xlabel('Cantidad')
plt.ylabel('Tipo')
plt.show()

### Interpretación de la distribución

La gráfica indica qué tipos reconoce con mayor frecuencia el modelo. Si detecta lugares pero no pedidos, no debemos concluir que no existan pedidos: probablemente el formato de pedido es específico del negocio y requiere una regla adicional.

## 6. Agregar reglas de negocio

Las expresiones regulares son útiles cuando un dato sigue un patrón estable. En este ejercicio suponemos que los pedidos tienen entre cinco y seis dígitos, los teléfonos diez dígitos y los importes aparecen con símbolo de moneda o con la palabra pesos.

In [ ]:
patrones = {
    'PEDIDO': r'(?<!\d)\d{5,6}(?!\d)',
    'TELEFONO': r'(?<!\d)\d{10}(?!\d)',
    'IMPORTE': r'(?:\$\s?\d[\d,]*(?:\.\d{2})?|\d[\d,]*(?:\.\d{2})?\s+pesos)'
}

def extraer_reglas(texto):
    resultados = []
    for tipo, patron in patrones.items():
        for coincidencia in re.finditer(patron, texto, flags=re.IGNORECASE):
            resultados.append({
                'entidad': coincidencia.group(),
                'tipo': tipo,
                'inicio': coincidencia.start(),
                'fin': coincidencia.end(),
                'origen': 'regla_negocio'
            })
    return resultados

df['entidades_reglas'] = df['texto'].apply(extraer_reglas)
display(df[['id_reporte', 'texto', 'entidades_reglas']].head(8))

### Interpretación de las reglas

Una regla puede ser muy precisa cuando el formato está controlado, pero también puede producir falsos positivos si es demasiado amplia. Por ejemplo, una regla de números debe distinguir un pedido de una fecha o un teléfono.

Las reglas deben probarse con ejemplos positivos y negativos, documentarse y actualizarse cuando cambien los formatos del negocio.

### ¿Cuándo usar modelo y cuándo usar reglas?

Use el modelo cuando la entidad depende del lenguaje y puede expresarse de muchas maneras: personas, empresas, lugares o fechas. Use reglas cuando el dato tiene un formato controlado: folios, teléfonos, códigos postales o importes.

La combinación suele ser más práctica que elegir una sola técnica: el modelo aporta generalización y las reglas aportan precisión sobre los datos críticos del negocio.

## 7. Unificar modelo y reglas

La arquitectura final combina flexibilidad y control: spaCy detecta entidades lingüísticas generales y las reglas detectan formatos propios de la empresa.

In [ ]:
def extraer_todo(texto):
    entidades = extraer_entidades_modelo(texto)
    entidades.extend(extraer_reglas(texto))
    return entidades

df['entidades_finales'] = df['texto'].apply(extraer_todo)
filas_finales = []
for _, fila in df.iterrows():
    for entidad in fila['entidades_finales']:
        filas_finales.append({'id_reporte': fila['id_reporte'], **entidad})

entidades_finales_df = pd.DataFrame(filas_finales)
entidades_finales_df.to_csv('entidades_extraidas.csv', index=False, encoding='utf-8')
display(entidades_finales_df.head(15))

## 8. Revisión de resultados

La siguiente celda permite comprobar si los pedidos, teléfonos e importes fueron recuperados por las reglas. También muestra todas las entidades del primer reporte para revisar el resultado de forma comprensible.

In [ ]:
for tipo in ['PEDIDO', 'TELEFONO', 'IMPORTE']:
    total = (entidades_finales_df['tipo'] == tipo).sum()
    print(f'{tipo}: {total} entidades')

print('\nEntidades del reporte R001:')
display(entidades_finales_df[entidades_finales_df['id_reporte'] == 'R001'])

### Interpretación de resultados

Si R001 muestra la persona, la empresa, la ciudad, la fecha y el pedido, el sistema logró convertir una oración en varios campos estructurados. La columna origen permite saber si cada resultado fue producido por el modelo o por una regla.

La tabla exportada puede utilizarse como insumo para un CRM, una base de datos, un tablero o una búsqueda por pedido y ciudad.

### Interpretación desde el punto de vista operativo

A partir de la tabla estructurada se pueden crear indicadores como reportes por ciudad, importes reclamados, pedidos pendientes o empresas con mayor número de incidencias. La extracción no resuelve por sí sola el problema operativo; prepara los datos para tomar decisiones y automatizar pasos posteriores.

Por ejemplo, un sistema podría identificar un pedido, consultar su estado en otra base de datos y mostrar al agente una respuesta sugerida. Ese flujo debe conservar trazabilidad: mensaje original, entidad extraída, regla o modelo usado y corrección humana.

## 9. Evaluación y limitaciones

Este notebook hace una revisión funcional, pero no una evaluación estadística completa. Para calcular precision, recall y F1 se necesita una muestra anotada manualmente con las entidades correctas.

En NER importa tanto detectar la entidad como delimitarla correctamente. También debemos revisar nombres desconocidos, abreviaturas, errores ortográficos, formatos nuevos y datos personales.

Antes de producción conviene:

- crear un conjunto de prueba anotado;
- medir cada tipo de entidad por separado;
- revisar falsos positivos y falsos negativos;
- proteger nombres, teléfonos y datos financieros;
- monitorear cambios en los formatos de pedidos e importes;
- permitir corrección humana y reentrenamiento.

## 10. Conclusiones generales

1. NER responde qué datos relevantes aparecen dentro de un texto y dónde se encuentran.
2. Un modelo lingüístico general reconoce patrones del idioma, pero no conoce automáticamente los códigos internos de una empresa.
3. Las reglas de negocio complementan al modelo cuando existen formatos estables como números de pedido, teléfonos e importes.
4. La salida útil es una tabla estructurada con reporte, entidad, tipo, posición y origen.
5. La automatización debe comenzar con apoyo al usuario, revisión de casos dudosos y medición de calidad.
6. La decisión de producción debe basarse en precision, recall, costo de errores, privacidad y beneficio operativo.

### Conclusión ejecutiva

La combinación de NER y reglas específicas convierte reportes de clientes en datos accionables. Es una técnica adecuada para extraer información, alimentar sistemas y reducir captura manual, siempre que se valide con datos reales y se controle la información sensible.